# Kia Bhubaneswar Dealership - Lead Conversion Intelligence
This notebook performs exploratory data analysis (EDA) and generates key visualizations for the Kia Bhubaneswar CRM lead dataset (Jan-Jun 2026).

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.family"] = "sans-serif"

## 1. Load Cleaned Dataset

In [ ]:
df = pd.read_csv('../data/master_clean.csv')
# Convert dates to datetime objects
date_cols = ['Enquiry Date', 'Test Drive Date', 'Booking Date', 'Retail date', 'Lost Date']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')
print(f"Dataset shape: {df.shape}")
df.head()

## 2. Setup Funnel Metrics

In [ ]:
# Set conversion flags
df['Is_Enquiry'] = True
df['Is_TD'] = df['Test Drive Date'].notnull() | (df['TD Status'] == 'Done')
df['Is_Booking'] = df['Booking Date'].notnull()
df['Is_Retail'] = df['Retail date'].notnull() | (df['Enquiry Status'] == 'Retail')

stages = ['Enquiry', 'Test Drive', 'Booking', 'Retail']
counts = [df['Is_Enquiry'].sum(), df['Is_TD'].sum(), df['Is_Booking'].sum(), df['Is_Retail'].sum()]

funnel_df = pd.DataFrame({'Stage': stages, 'Leads': counts})
funnel_df['Conversion_Rate_Pct'] = (funnel_df['Leads'] / len(df) * 100).round(2)
funnel_df

In [ ]:
# Visualizing the Sales Funnel
plt.figure(figsize=(10, 5))
sns.barplot(x='Stage', y='Leads', data=funnel_df, palette='Reds_r')
plt.title('Kia Bhubaneswar Sales Funnel (Jan-Jun 2026)', fontsize=14, fontweight='bold', color='#333333')
for i, val in enumerate(funnel_df['Leads']):
    pct = funnel_df['Conversion_Rate_Pct'].iloc[i]
    plt.text(i, val + 150, f"{val:,} ({pct}%)", ha='center', fontsize=11, fontweight='bold')
plt.ylim(0, 8000)
plt.ylabel('Number of Leads')
plt.xlabel('Funnel Stage')
plt.tight_layout()
plt.savefig('funnel_plot.png')
plt.show()

## 3. Lead Source Conversion Rates

In [ ]:
source_perf = df.groupby('Source').agg(
    Enquiries=('Is_Enquiry', 'sum'),
    Retails=('Is_Retail', 'sum')
).reset_index()
source_perf['Retail_Conversion_Rate_Pct'] = (source_perf['Retails'] / source_perf['Enquiries'] * 100).round(2)
source_perf = source_perf.sort_values(by='Retail_Conversion_Rate_Pct', ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(x='Retail_Conversion_Rate_Pct', y='Source', data=source_perf, palette='viridis')
plt.title('Retail Conversion Rate by Lead Source', fontsize=14, fontweight='bold')
plt.xlabel('Conversion Rate (%)')
plt.ylabel('Sourcing Channel')
for i, row in enumerate(source_perf.itertuples()):
    plt.text(row.Retail_Conversion_Rate_Pct + 0.5, i, f"{row.Retail_Conversion_Rate_Pct}% (N={row.Enquiries})", va='center', fontweight='bold')
plt.xlim(0, 32)
plt.tight_layout()
plt.savefig('source_conversion_plot.png')
plt.show()

## 4. Test Drive (TD) Impact

In [ ]:
td_impact = df.groupby('Is_TD').agg(
    Enquiries=('Is_Enquiry', 'sum'),
    Bookings=('Is_Booking', 'sum'),
    Retails=('Is_Retail', 'sum')
).reset_index()
td_impact['Booking_Rate_Pct'] = (td_impact['Bookings'] / td_impact['Enquiries'] * 100).round(2)
td_impact['Retail_Rate_Pct'] = (td_impact['Retails'] / td_impact['Enquiries'] * 100).round(2)
td_impact['Is_TD'] = td_impact['Is_TD'].map({False: 'No Test Drive', True: 'Test Drive Completed'})
td_impact

In [ ]:
# Plotting TD Impact
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(x='Is_TD', y='Booking_Rate_Pct', data=td_impact, ax=ax[0], palette='Blues')
ax[0].set_title('Booking Rate by Test Drive Status', fontsize=13, fontweight='bold')
ax[0].set_ylabel('Booking Rate (%)')
ax[0].set_xlabel('')
for i, val in enumerate(td_impact['Booking_Rate_Pct']):
    ax[0].text(i, val + 1, f"{val}%", ha='center', fontweight='bold', fontsize=12)

sns.barplot(x='Is_TD', y='Retail_Rate_Pct', data=td_impact, ax=ax[1], palette='Oranges')
ax[1].set_title('Retail Delivery Rate by Test Drive Status', fontsize=13, fontweight='bold')
ax[1].set_ylabel('Retail Conversion Rate (%)')
ax[1].set_xlabel('')
for i, val in enumerate(td_impact['Retail_Rate_Pct']):
    ax[1].text(i, val + 0.5, f"{val}%", ha='center', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('td_impact_plot.png')
plt.show()

## 5. Follow-up Optimization analysis

In [ ]:
# Grouping by Followup Count
followup_counts = df.groupby('Followup Count').agg(
    Leads=('Is_Enquiry', 'sum'),
    Retails=('Is_Retail', 'sum')
).reset_index()
followup_counts['Retail_Conversion_Rate_Pct'] = (followup_counts['Retails'] / followup_counts['Leads'] * 100).round(2)

# Filter to counts with at least 5 leads to avoid extreme noise in visualization
followup_filtered = followup_counts[followup_counts['Leads'] >= 5].head(10)

fig, ax1 = plt.subplots(figsize=(12, 6))
color = 'tab:blue'
ax1.set_xlabel('Follow-up Call Count', fontweight='bold')
ax1.set_ylabel('Number of Leads', color=color, fontweight='bold')
sns.barplot(x='Followup Count', y='Leads', data=followup_filtered, ax=ax1, color='lightblue')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Retail Conversion Rate (%)', color=color, fontweight='bold')
sns.lineplot(x='Followup Count', y='Retail_Conversion_Rate_Pct', data=followup_filtered, ax=ax2, color=color, marker='o', linewidth=2.5)
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Follow-up S curve vs Lead Volume', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('followup_optimization_plot.png')
plt.show()